# Ex1 - Filtering and Sorting Data



### Step 1. Import the necessary libraries

In [1]:
import pandas as pd
import numpy as np

### Step 2. Import the dataset from this [address](https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv) and assign it to a variable called chipo.

In [2]:
url = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv"
chipo = pd.read_csv(url, sep="\t")
chipo

,order_id,quantity,item_name,choice_description,item_price
0,1,1,Chips and Fresh Tomato Salsa,NaN,$2.39
1,1,1,Izze,[Clementine],$3.39
2,1,1,Nantucket Nectar,[Apple],$3.39
3,1,1,Chips and Tomatillo-Green Chili Salsa,NaN,$2.39
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",$16.98
...,...,...,...,...,...
4617,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Black Beans, Sour ...",$11.75
4618,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Sour Cream, Cheese...",$11.75
4619,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Pinto...",$11.25
4620,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Lettu...",$8.75


In [6]:
chipo.info()

<class 'pandas.DataFrame'>
RangeIndex: 4622 entries, 0 to 4621
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   order_id            4622 non-null   int64  
 1   quantity            4622 non-null   int64  
 2   item_name           4622 non-null   str    
 3   choice_description  3376 non-null   str    
 4   item_price          4622 non-null   float64
dtypes: float64(1), int64(2), str(2)
memory usage: 180.7 KB


In [8]:
chipo.describe().T

,count,mean,std,min,25%,50%,75%,max
order_id,4622.0,927.254868,528.890796,1.00,477.25,926.00,1393.00,1834.00
quantity,4622.0,1.075725,0.410186,1.00,1.00,1.00,1.00,15.00
item_price,4622.0,7.464336,4.245557,1.09,3.39,8.75,9.25,44.25


### Step 3. Name of the max valued product

In [3]:
print(chipo.dtypes)

order_id              int64
quantity              int64
item_name               str
choice_description      str
item_price              str
dtype: object


In [4]:
# Name of the max valued product:
# Primero a tener en cuenta, item_price es un string, por lo que hay que convertirlo a float:
chipo["item_price"] = chipo["item_price"].str.replace("$", "").astype(float)

In [5]:
max_item = chipo.loc[chipo["item_price"].idxmax(), "item_name"]
max_item

'Chips and Fresh Tomato Salsa'

In [ ]:
prod_caro = chipo.loc[(chipo["item_price"]/chipo["quantity"]).idxmax(), "item_name"]
print(prod_caro)
# Asi sí, se obtiene el producto más caro por unidad, no el más caro en total.

Steak Salad Bowl


In [12]:
chipo["precio_unitario"] = chipo["item_price"] / chipo["quantity"]

In [13]:
chipo.head()

,order_id,quantity,item_name,choice_description,item_price,precio_unitario
0,1,1,Chips and Fresh Tomato Salsa,NaN,2.39,2.39
1,1,1,Izze,[Clementine],3.39,3.39
2,1,1,Nantucket Nectar,[Apple],3.39,3.39
3,1,1,Chips and Tomatillo-Green Chili Salsa,NaN,2.39,2.39
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",16.98,8.49


### Step 4. How many products cost more than $10.00?

In [25]:
# How many products cost more than $10.00?
count = (chipo["item_price"] > 10.00).sum()
print(f"Número de productos con precio mayor a $10: {count}")

Número de productos con precio mayor a $10: 1130


In [33]:
chipo[chipo["precio_unitario"] > 10.00]["item_name"].nunique()
#print(f"Número de productos con precio unitario mayor a $10: {count_unitario}")

25

### Step 4.1: Y cuántos pedidos se han hecho con un producto de más de 10$? Es lo mismo?

In [30]:
# Cuántos pedidos se han hecho con un producto de más de $10.00?
expensive_orders = chipo[chipo["item_price"] > 10]["order_id"].nunique()
print(f"Número de pedidos con al menos un producto de más de $10: {expensive_orders}")

'''No, no es lo mismo que el anterior, puesto que el anterior cuenta los pedidos, y este los PRODUCTOS. 
Un pedido puede tener varios productos, y no todos tienen por qué tener un precio mayor de 10 dólares.'''

Número de pedidos con al menos un producto de más de $10: 863


'No, no es lo mismo que el anterior, puesto que el anterior cuenta los pedidos, y este los PRODUCTOS. \nUn pedido puede tener varios productos, y no todos tienen por qué tener un precio mayor de 10 dólares.'

In [15]:
expensive_orders_unique = chipo[chipo["precio_unitario"] > 10]["order_id"].nunique()
print(f"Número de pedidos con al menos un producto con precio unitario mayor a $10: {expensive_orders_unique}")

Número de pedidos con al menos un producto con precio unitario mayor a $10: 787


### Step 4.2: Y cuántos pedidos se han hecho de más de 10$? Es lo mismo?

In [31]:
# Cuántos pedidos se han hecho de más de $10.00?
expensive_orders_total = chipo.groupby("order_id")["item_price"].sum()
expensive_orders_total = expensive_orders_total[expensive_orders_total > 10].count()
print(f"Número de pedidos con un total mayor a $10: {expensive_orders_total}")

'''No es lo mismo, puesto que este cuenta los PEDIDOS cuyo total es mayor a 10 dólares, 
mientras que el anterior contaba los pedidos que tenían al menos un PRODUCTO de más de 10 dólares.'''

Número de pedidos con un total mayor a $10: 1834


'No es lo mismo, puesto que este cuenta los PEDIDOS cuyo total es mayor a 10 dólares, \nmientras que el anterior contaba los pedidos que tenían al menos un PRODUCTO de más de 10 dólares.'

### Step 4.3: Y en cuántos pedidos se ha pagado más de 10$ por un mismo producto? Es lo mismo?

In [ ]:
# En cuántos pedidos se ha pagado más de $10.00 por un mismo producto?
chipo["product_total"] = chipo["quantity"] * chipo["item_price"]
expensive_orders = chipo[chipo["product_total"] > 10]
expensive_orders_count = expensive_orders["order_id"].nunique()
print(f"Número de pedidos con un mismo producto que cuesta más de $10: {expensive_orders_count}")
# No es lo mismo y ahí lo dejo ahora, estoy cansada, pero vamos NUMERO de PEDIDOS con mas de un producto que valga mas de 10 dólares.

Número de pedidos con un mismo producto que cuesta más de $10: 889


### Step 5. What is the price of each item and name it unit_price. Get only item_name and unit_price

In [45]:
# What is the price of each item and name it unit_price?
chipo["unit_price"] = chipo["item_price"] / chipo["quantity"]
chipo[["item_name", "unit_price"]]

,item_name,unit_price
0,Chips and Fresh Tomato Salsa,2.39
1,Izze,3.39
2,Nantucket Nectar,3.39
3,Chips and Tomatillo-Green Chili Salsa,2.39
4,Chicken Bowl,8.49
...,...,...
4617,Steak Burrito,11.75
4618,Steak Burrito,11.75
4619,Chicken Salad Bowl,11.25
4620,Chicken Salad Bowl,8.75


### Step 6. Sort by the name of the item

In [46]:
# Sort by the name of the item:
chipo.sort_values("item_name")[["item_name", "item_price"]]

,item_name,item_price
3389,6 Pack Soft Drink,12.98
341,6 Pack Soft Drink,6.49
1849,6 Pack Soft Drink,6.49
1860,6 Pack Soft Drink,6.49
2713,6 Pack Soft Drink,6.49
...,...,...
2384,Veggie Soft Tacos,8.75
781,Veggie Soft Tacos,8.75
2851,Veggie Soft Tacos,8.49
1699,Veggie Soft Tacos,11.25


### Step 7. What was the quantity of the most expensive item ordered? 2 ways

In [34]:
# What was the quantity of the most expensive item ordered?
most_expensive_item = chipo.loc[chipo["item_price"].idxmax()]
quantity_most_expensive = most_expensive_item["quantity"]
print(f"Cantidad del producto más caro: {quantity_most_expensive}")

Cantidad del producto más caro: 15


In [52]:
# Otra manera:
most_expensive = chipo.sort_values("item_price", ascending=False)
quantity = most_expensive.iloc[0]["quantity"]

print(f"Cantidad del producto más caro: {quantity}")

Cantidad del producto más caro: 15


### Step 8. How many times was a Veggie Salad Bowl ordered?

In [47]:
# How many times was a Veggie Salad Bowl ordered?
veggie_salad_bowl_count = chipo[chipo["item_name"] == "Veggie Salad Bowl"]["quantity"].sum()
print(f"Cantidad total de Veggie Salad Bowl pedidos: {veggie_salad_bowl_count}")

Cantidad total de Veggie Salad Bowl pedidos: 18


### Step 9. How many times did someone order more than one Canned Soda?

In [ ]:
# How many times did someone order more than on canned soda?
soda_count = chipo[(chipo["item_name"] == "Canned Soda") & (chipo["quantity"] > 1)].shape[0]
print(f"Cantidad de veces que se pidió más de una lata de soda: {soda_count}")

Cantidad de veces que se ordenó más de una lata de soda: 20
